<a href="https://colab.research.google.com/github/Tokhirjonov15/Menu_detector_AI/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print('Hello World!')

Hello World!


In [2]:
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError
import os
import numpy as np

from torchvision.models import mobilenet_v2

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_Path:', DATASET_PATH)

CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert',  # Label Grouping | Class Consolidation
    'cheesecake': 'dessert',      # Label Grouping | Class Consolidation
    'kebab': 'kebab',
    'pilaf': 'pilaf'
}

CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab', 'pilaf']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print(NUM_CLASSES)
print(CLASS_TO_IDX)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Dataset_Path: /content/drive/MyDrive/food101_dataset
5
{'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab': 3, 'pilaf': 4}


In [30]:
# Custom Dataset Class

class FoodDataset(Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    # print('Images_Length', len(self.images))
    return len(self.images)

  def __getitem__(self, idx):
    img_path = self.images[idx]
    # print('Image_Path', img_path)
    label = self.labels[idx]
    # print('Label', label)
    try:
      image = Image.open(img_path)
      if image.mode == 'P' or image.mode == "RGBA":
        image = image.convert('RGBA').convert('RGB')
      else:
        image = image.convert('RGB')
    except (UnidentifiedImageError, OSError):
      print(f"Skipping broken image: {img_path}")
      return self.__getitem__((idx + 1) % len(self.images))
    if self.transform:
      image = self.transform(image)
    return image, label

In [31]:
# Gather and Split Data

all_images = []
for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
  class_path = os.path.join(DATASET_PATH, original_class)
  print('Class_Path', class_path)
  if not os.path.exists(class_path):
    print(f"Warning: {class_path} not found")
    continue
  for img in os.listdir(class_path):
    if not img.endswith(('.jpg', '.jpeg', '.png')):
      full_path = os.path.join(class_path, img)
      all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)
split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

print("All_Images:", all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))
img, lbl = dataset[0]

Class_Path /content/drive/MyDrive/food101_dataset/hamburger
Class_Path /content/drive/MyDrive/food101_dataset/hot_dog
Class_Path /content/drive/MyDrive/food101_dataset/chocolate_cake
Class_Path /content/drive/MyDrive/food101_dataset/cheesecake
Class_Path /content/drive/MyDrive/food101_dataset/kebab
Class_Path /content/drive/MyDrive/food101_dataset/pilaf
All_Images: [('/content/drive/MyDrive/food101_dataset/kebab/Image_58.webp', 3), ('/content/drive/MyDrive/food101_dataset/kebab/Image_46.webp', 3), ('/content/drive/MyDrive/food101_dataset/kebab/Image_39.webp', 3)]
2


In [32]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [33]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2) # thread | parallel loading for speed
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [34]:
# pretrained model
model = mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES) # fine-tuning

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)
model = model.to(device)

device: cuda


In [37]:
criterion = nn.CrossEntropyLoss() # Loss Function  |  '70% burger', '30% hot dog'
optimizer = optim.Adam(model.parameters(), lr=0.001) # Weight
torch.backends.cudnn.benchmark = True  # Benchmark Setting | Trick

In [39]:
# Training Loop

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
  model.train()
  running_loss = 0.0
  for images, labels in train_loader:  # forward and backward(backpropagation)
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()  # zero the gradient
    outputs = model(images)
    loss = criterion(outputs, labels)  # calculate loss
    loss.backward()
    optimizer.step()  # Adam optimizer
    running_loss += loss.item()  # track loss

  # Validation
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for images, labels in val_loader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      __, predicted = torch.max(outputs, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  val_acc = 100 * correct / total
  print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}], Loss: {running_loss / len(train_loader):.4f}, Validation Accuracy: {val_acc:.2f}%")

  if val_acc > best_accuracy:
    best_accuracy = val_acc
    torch.save(model.state_dict(), '/content/menu_detector.pth')
    print("Saved new best model!")

Epoch [1/10], Loss: 0.0007, Validation Accuracy: 100.00%
Saved new best model!
Epoch [2/10], Loss: 0.0003, Validation Accuracy: 100.00%
Epoch [3/10], Loss: 0.0003, Validation Accuracy: 100.00%
Epoch [4/10], Loss: 0.0002, Validation Accuracy: 100.00%
Epoch [5/10], Loss: 0.0002, Validation Accuracy: 100.00%
Epoch [6/10], Loss: 0.0001, Validation Accuracy: 100.00%
Epoch [7/10], Loss: 0.0001, Validation Accuracy: 100.00%
Epoch [8/10], Loss: 0.0001, Validation Accuracy: 100.00%
Epoch [9/10], Loss: 0.0000, Validation Accuracy: 100.00%
Epoch [10/10], Loss: 0.0000, Validation Accuracy: 100.00%
